# 🔐 Detecção de Ameaças Cibernéticas com Redes Neurais (MLP)

**Dataset:** [Cybersecurity Threat Detection Dataset](https://www.kaggle.com/datasets/dhrubangtalukdar/cybersecurity-threat-detection-dataset) — Kaggle  
**Ambiente:** Python 3.10 · PyTorch · scikit-learn · Google Colab / Jupyter

---

## 📋 Sumário

| # | Etapa | Descrição |
|---|-------|-----------|
| 1 | [Importações e Configurações](#passo-1) | Bibliotecas, seed e dispositivo |
| 2 | [Carregamento e Exploração](#passo-2) | Download, shape, info e nulos |
| 3 | [Pré-processamento](#passo-3) | Limpeza, encoding e normalização |
| 4 | [Definição do Modelo](#passo-4) | Arquitetura MLP (PyTorch) |
| 5 | [Treinamento](#passo-5) | Loop de treino com histórico |
| 6 | [Avaliação de Performance](#passo-6) | Curvas, acurácia e matriz de confusão |
| 7 | [Reflexão Final](#passo-7) | Análise crítica dos resultados |

---

> **Objetivo:** Treinar um classificador de rede neural do tipo *Multilayer Perceptron* (MLP) capaz de identificar diferentes categorias de ataques em tráfego de rede, utilizando features extraídas de logs de segurança.


---
<a id="passo-1"></a>
## 1 · Importações e Configurações

In [ ]:
# ─── Instale dependências se necessário ────────────────────────────────────
# !pip install torch pandas scikit-learn matplotlib seaborn kagglehub

import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

warnings.filterwarnings("ignore")

# ─── Reprodutibilidade ──────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ─── Dispositivo ────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Dispositivo em uso: {device}")
print(f"   PyTorch version : {torch.__version__}")
print(f"   CUDA disponível : {torch.cuda.is_available()}")


---
<a id="passo-2"></a>
## 2 · Carregamento e Exploração do Dataset

> **Fonte:** Kaggle — *Cybersecurity Threat Detection Dataset*  
> Contém registros de tráfego de rede com features como protocolo, flags, tamanho de pacotes e tipo de ataque.


In [ ]:
import kagglehub

# ─── Download via kagglehub ─────────────────────────────────────────────────
path = kagglehub.dataset_download("dhrubangtalukdar/cybersecurity-threat-detection-dataset")
print(f"📂 Caminho do dataset: {path}")

arquivos = os.listdir(path)
print("\nArquivos disponíveis:")
for f in arquivos:
    print(f"  └─ {f}")


In [ ]:
# ─── Leitura do CSV ─────────────────────────────────────────────────────────
csv_file = next(f for f in os.listdir(path) if f.endswith(".csv"))
df = pd.read_csv(os.path.join(path, csv_file))

print(f"📄 Arquivo carregado : {csv_file}")
print(f"📐 Dimensões         : {df.shape[0]:,} linhas × {df.shape[1]} colunas")
df.head()


In [ ]:
# ─── Informações gerais ─────────────────────────────────────────────────────
print("=" * 55)
print("INFORMAÇÕES DO DATASET")
print("=" * 55)
df.info()


In [ ]:
# ─── Estatísticas descritivas ───────────────────────────────────────────────
print("Estatísticas Descritivas — features numéricas")
df.describe().round(4)


In [ ]:
# ─── Valores nulos ──────────────────────────────────────────────────────────
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(2)
resumo_nulos = pd.DataFrame({"Nulos": nulos, "Percentual (%)": nulos_pct})
resumo_nulos = resumo_nulos[resumo_nulos["Nulos"] > 0]

if resumo_nulos.empty:
    print("✅ Nenhum valor nulo encontrado no dataset.")
else:
    print("⚠️  Colunas com valores nulos:")
    print(resumo_nulos)


In [ ]:
# ─── Coluna alvo (target) ───────────────────────────────────────────────────
# Ajuste o nome abaixo caso o dataset use outro rótulo
TARGET_COL = "Attack Type"

print(f"🎯 Coluna alvo: '{TARGET_COL}'")
print(f"   Classes distintas: {df[TARGET_COL].nunique()}")
print("\nDistribuição das classes:")
print(df[TARGET_COL].value_counts().to_string())

# Gráfico de distribuição
fig, ax = plt.subplots(figsize=(9, 4))
contagens = df[TARGET_COL].value_counts()
contagens.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white", linewidth=0.5)
ax.set_title("Distribuição das Classes (variável alvo)", fontsize=13, pad=12)
ax.set_xlabel("Tipo de Ataque", labelpad=8)
ax.set_ylabel("Número de Amostras")
ax.tick_params(axis="x", rotation=35)
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + contagens.max() * 0.01,
            f"{int(bar.get_height()):,}", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.show()


---
<a id="passo-3"></a>
## 3 · Pré-processamento dos Dados

Pipeline adotado:

1. Remoção de colunas não informativas (IPs, timestamps, portas)  
2. Tratamento de valores nulos (mediana / moda)  
3. Encoding de variáveis categóricas com `LabelEncoder`  
4. Separação de features (X) e target (y)  
5. Normalização com `StandardScaler`  
6. Divisão treino / teste (80 / 20)


In [ ]:
# ─── 3.1 · Remoção de colunas não informativas ──────────────────────────────
COLUNAS_REMOVER = [
    "Source IP", "Destination IP", "Timestamp",
    "Payload Data", "Source Port", "Destination Port",
]
colunas_encontradas = [c for c in COLUNAS_REMOVER if c in df.columns]

df_clean = df.drop(columns=colunas_encontradas)
print(f"🗑️  Colunas removidas ({len(colunas_encontradas)}): {colunas_encontradas}")
print(f"📐 Shape após remoção: {df_clean.shape}")


In [ ]:
# ─── 3.2 · Tratamento de valores nulos ──────────────────────────────────────
for col in df_clean.columns:
    qtd_nulos = df_clean[col].isnull().sum()
    if qtd_nulos > 0:
        if df_clean[col].dtype == "object":
            df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)
            estrategia = "moda"
        else:
            df_clean[col].fillna(df_clean[col].median(), inplace=True)
            estrategia = "mediana"
        print(f"  [{col}] → {qtd_nulos} nulos preenchidos com {estrategia}")

total_restante = df_clean.isnull().sum().sum()
print(f"\n✅ Valores nulos restantes: {total_restante}")


In [ ]:
# ─── 3.3 · Encoding de variáveis categóricas (exceto target) ────────────────
label_encoders = {}

for col in df_clean.columns:
    if df_clean[col].dtype == "object" and col != TARGET_COL:
        le = LabelEncoder()
        df_clean[col] = le.fit_transform(df_clean[col].astype(str))
        label_encoders[col] = le

print(f"🔢 Colunas codificadas ({len(label_encoders)}): {list(label_encoders.keys())}")


In [ ]:
# ─── 3.4 · Separação features / target ──────────────────────────────────────
X = df_clean.drop(columns=[TARGET_COL]).values
y_raw = df_clean[TARGET_COL].values

le_target = LabelEncoder()
y = le_target.fit_transform(y_raw)

print(f"📊 Classes detectadas : {list(le_target.classes_)}")
print(f"   X shape: {X.shape}  |  y shape: {y.shape}")


In [ ]:
# ─── 3.5 · Normalização (StandardScaler) ────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("✅ Normalização aplicada com StandardScaler")
print(f"   Média   (5 primeiras features): {X_scaled[:, :5].mean(axis=0).round(4)}")
print(f"   Std Dev (5 primeiras features): {X_scaled[:, :5].std(axis=0).round(4)}")


In [ ]:
# ─── 3.6 · Split Treino / Teste (80% / 20%) ────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f"📦 Treino : {X_train.shape[0]:,} amostras  ({X_train.shape[0]/len(X_scaled)*100:.0f}%)")
print(f"📦 Teste  : {X_test.shape[0]:,} amostras  ({X_test.shape[0]/len(X_scaled)*100:.0f}%)")

# Tensores PyTorch
X_train_t = torch.FloatTensor(X_train).to(device)
X_test_t  = torch.FloatTensor(X_test).to(device)
y_train_t = torch.LongTensor(y_train).to(device)
y_test_t  = torch.LongTensor(y_test).to(device)

# DataLoaders
BATCH_SIZE = 64
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test_t,  y_test_t),  batch_size=BATCH_SIZE, shuffle=False)

print(f"\n⚙️  Batch size    : {BATCH_SIZE}")
print(f"   Batches/época : {len(train_loader)}")
print(f"   Num. classes  : {len(le_target.classes_)}")


---
<a id="passo-4"></a>
## 4 · Definição do Modelo de Rede Neural

Arquitetura MLP com camadas totalmente conectadas:

```
Input(n_features)
    └─► Linear(128) → BatchNorm → ReLU → Dropout(0.3)
        └─► Linear(64)  → BatchNorm → ReLU → Dropout(0.3)
            └─► Linear(32)  → BatchNorm → ReLU
                └─► Linear(num_classes)   [logits]
```

| Técnica | Motivo |
|---------|--------|
| **BatchNorm** | Estabiliza o treinamento e acelera convergência |
| **ReLU** | Ativação não-linear; evita gradientes que desaparecem |
| **Dropout (30%)** | Regularização; reduz overfitting |
| **CrossEntropyLoss** | Adequada para classificação multiclasse |
| **Adam** | Otimizador adaptativo; bom desempenho padrão |


In [ ]:
class CybersecurityMLP(nn.Module):
    """
    Multilayer Perceptron para classificação de ameaças cibernéticas.

    Parâmetros
    ----------
    input_size  : int — número de features de entrada
    num_classes : int — número de classes de saída
    """

    def __init__(self, input_size: int, num_classes: int):
        super().__init__()
        self.network = nn.Sequential(
            # Camada 1
            nn.Linear(input_size, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            # Camada 2
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            # Camada 3
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            # Saída
            nn.Linear(32, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


# ─── Instanciar ─────────────────────────────────────────────────────────────
INPUT_SIZE  = X_train.shape[1]
NUM_CLASSES = len(le_target.classes_)

model = CybersecurityMLP(INPUT_SIZE, NUM_CLASSES).to(device)

# Parâmetros treináveis
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\n🔢 Parâmetros treináveis : {n_params:,}")
print(f"   Input size           : {INPUT_SIZE}")
print(f"   Num classes          : {NUM_CLASSES}")


In [ ]:
# ─── Critério de perda e otimizador ─────────────────────────────────────────
LEARNING_RATE = 1e-3

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"⚙️  Critério  : CrossEntropyLoss")
print(f"   Otimizador : Adam  |  lr = {LEARNING_RATE}")


---
<a id="passo-5"></a>
## 5 · Treinamento do Modelo


In [ ]:
def evaluate(model: nn.Module, loader: DataLoader):
    """Calcula loss e acurácia no conjunto passado (sem atualizar pesos)."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            outputs = model(X_batch)
            total_loss += criterion(outputs, y_batch).item()
            preds = outputs.argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)
    return total_loss / len(loader), correct / total


In [ ]:
NUM_EPOCHS = 30

history = {k: [] for k in ("train_loss", "train_acc", "val_loss", "val_acc")}

print(f"{'Época':>6}  {'Train Loss':>10}  {'Train Acc':>9}  {'Val Loss':>9}  {'Val Acc':>8}")
print("─" * 55)

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    epoch_loss, correct, total = 0.0, 0, 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

    t_loss = epoch_loss / len(train_loader)
    t_acc  = correct / total
    v_loss, v_acc = evaluate(model, test_loader)

    history["train_loss"].append(t_loss)
    history["train_acc"].append(t_acc)
    history["val_loss"].append(v_loss)
    history["val_acc"].append(v_acc)

    if epoch % 5 == 0 or epoch == 1:
        print(f"{epoch:>6}  {t_loss:>10.4f}  {t_acc:>9.4f}  {v_loss:>9.4f}  {v_acc:>8.4f}")

print("─" * 55)
print("\n✅ Treinamento concluído!")


---
<a id="passo-6"></a>
## 6 · Avaliação de Performance


In [ ]:
# ─── 6.1 · Curvas de Loss e Acurácia ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epocas = range(1, NUM_EPOCHS + 1)

for ax, metrica, titulo, ylabel in zip(
    axes,
    [("train_loss", "val_loss"), ("train_acc", "val_acc")],
    ["Curva de Perda (Loss)", "Curva de Acurácia"],
    ["Loss", "Acurácia"],
):
    ax.plot(epocas, history[metrica[0]], label="Treino",    color="steelblue",  linewidth=2)
    ax.plot(epocas, history[metrica[1]], label="Validação", color="tomato",     linewidth=2, linestyle="--")
    ax.set_title(titulo, fontsize=13, pad=10)
    ax.set_xlabel("Época")
    ax.set_ylabel(ylabel)
    ax.legend(framealpha=0.8)
    ax.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()


In [ ]:
# ─── 6.2 · Métricas finais ──────────────────────────────────────────────────
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        preds = model(X_batch).argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_batch.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
print(f"🏆 Acurácia Final no Conjunto de Teste: {acc:.4f}  ({acc*100:.2f}%)")
print("\n" + "=" * 55)
print("RELATÓRIO DE CLASSIFICAÇÃO")
print("=" * 55)
print(classification_report(all_labels, all_preds, target_names=le_target.classes_))


In [ ]:
# ─── 6.3 · Matriz de Confusão ───────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)
cm_norm = confusion_matrix(all_labels, all_preds, normalize="true")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, matriz, titulo, fmt in zip(
    axes,
    [cm,          cm_norm],
    ["Matriz de Confusão — Valores Absolutos",
     "Matriz de Confusão — Normalizada (por linha)"],
    ["d",         ".2f"],
):
    sns.heatmap(
        matriz, annot=True, fmt=fmt, cmap="Blues", ax=ax,
        xticklabels=le_target.classes_,
        yticklabels=le_target.classes_,
        linewidths=0.5, linecolor="white",
    )
    ax.set_title(titulo, fontsize=12, pad=10)
    ax.set_xlabel("Predito",  labelpad=8)
    ax.set_ylabel("Real",     labelpad=8)
    ax.tick_params(axis="x", rotation=35)

plt.tight_layout()
plt.show()


---
<a id="passo-7"></a>
## 7 · Reflexão sobre o Processo

### 7.1 · Decisões de Pré-processamento

O dataset de detecção de ameaças cibernéticas contém features de tráfego de rede — protocolos, flags, tamanhos de pacotes e tipos de ataque — que exigiram um pipeline de pré-processamento cuidadoso:

- **Remoção de colunas não informativas:** endereços IP, timestamps e portas foram descartados por serem identificadores que não generalizam para novos dados.
- **Tratamento de nulos:** valores faltantes em colunas numéricas foram substituídos pela mediana (robusta a outliers); em colunas categóricas, pela moda.
- **Encoding:** variáveis categóricas foram convertidas para representação numérica com `LabelEncoder`, necessário para o processamento pela rede neural.
- **Normalização:** o `StandardScaler` garantiu que todas as features tivessem média ≈ 0 e desvio padrão ≈ 1, evitando que features de maior magnitude dominem o gradiente.

### 7.2 · Decisões de Arquitetura

A MLP com 3 camadas ocultas (128 → 64 → 32 neurônios) foi escolhida por sua capacidade de aprender representações hierárquicas de features tabulares sem a complexidade de arquiteturas sequenciais (RNN) ou convolucionais (CNN). O uso de **BatchNorm** acelerou a convergência e o **Dropout (30%)** atuou como regularizador.

### 7.3 · Análise dos Resultados

> ✏️ **Preencha esta seção com base nos seus resultados reais:**

- **Acurácia final:** `XX%` *(substitua pelo valor obtido)*
- **Curvas de treinamento:** *(o modelo convergiu de forma estável? Houve sinal de overfitting — treino sobe mas validação estagna?)*
- **Matriz de confusão:** *(quais classes o modelo confundiu com mais frequência? Por quê?)*
- **Possíveis melhorias:** aumentar épocas, ajustar dropout, explorar outras arquiteturas (ex.: redes mais profundas, atenção tabular), ou balancear classes com técnicas como SMOTE.

---

*Projeto desenvolvido para a disciplina de Engenharia e Análise de Dados.*
